# UrduStack — Model Evaluation

Run the trained model against 107 hand-labeled test examples.
Computes precision, recall, F1, and accuracy.

**Two modes:**
- **Adversarial only**: 12 red-team cases (the original test suite)
- **Full evaluation**: all 107 examples across 5 categories

**Steps:**
1. Run cells 1-3 (setup + repo clone)
2. Upload model files in cell 4
3. Run cell 5 for adversarial tests (quick, ~1 minute)
4. Run cell 6 for full evaluation (~5-10 minutes)
5. Run cell 7 to view + download results

In [ ]:
import sys, torch

if not torch.cuda.is_available():
    print("WARNING: No GPU. Demo will work but slower.")
    print("For best results: Runtime > Change runtime type > T4 GPU")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch: {torch.__version__}")
    print("GPU check passed.")

In [ ]:
import subprocess, time

def pip_install(packages, attempt=1):
    print(f"Attempt {attempt}: installing packages...")
    subprocess.run(["pip", "uninstall", "-y", "torchao"], capture_output=True)
    result = subprocess.run(
        ["pip", "install", "-q", "-U"] + packages,
        capture_output=True, text=True
    )
    return result.returncode == 0

packages = [
    "transformers>=4.46.0",
    "datasets>=3.1.0",
    "peft>=0.13.2",
    "accelerate>=1.1.0",
    "openai-whisper>=20231117",
    "scikit-learn",
    "pandas",
]

ok = pip_install(packages, 1)
if not ok:
    print("First attempt failed, retrying...")
    time.sleep(5)
    ok = pip_install(packages, 2)
if not ok:
    raise RuntimeError("pip install failed")

import peft, transformers, datasets, sklearn
print(f"peft={peft.__version__}  transformers={transformers.__version__}  sklearn={sklearn.__version__}")
print("All dependencies ready.")

In [ ]:
import os, shutil, subprocess, time

if os.path.exists('UrduStack/UrduStack'):
    shutil.rmtree('UrduStack/UrduStack')

if os.path.exists('UrduStack'):
    os.chdir('UrduStack')
    print("Updating repo to latest code...")

    # Back up trained model files before reset (they're tracked by git
    # and will be overwritten with placeholder versions)
    backup_dir = '/tmp/urdustack_model_backup'
    if os.path.exists(backup_dir):
        shutil.rmtree(backup_dir)
    os.makedirs(backup_dir, exist_ok=True)

    has_trained_model = (
        os.path.exists('models/temperature.txt')
        and open('models/temperature.txt').read().strip()
        and (os.path.exists('models/risk_lora/adapter_model.safetensors')
             or os.path.exists('models/risk_lora/adapter_model.bin'))
    )

    if has_trained_model:
        shutil.copy('models/temperature.txt', backup_dir)
        shutil.copytree('models/risk_lora', os.path.join(backup_dir, 'risk_lora'),
                        dirs_exist_ok=True)
        print(f"  Backed up model files to {backup_dir}")

    subprocess.run(['git', 'fetch', 'origin'], capture_output=True)
    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"WARNING: git reset failed: {result.stderr}")
    else:
        print("Code updated to latest main.")

    # Restore trained model files over placeholders
    if has_trained_model:
        shutil.copy(os.path.join(backup_dir, 'temperature.txt'), 'models/temperature.txt')
        src_lora = os.path.join(backup_dir, 'risk_lora')
        dst_lora = 'models/risk_lora'
        os.makedirs(dst_lora, exist_ok=True)
        for fname in os.listdir(src_lora):
            shutil.copy2(os.path.join(src_lora, fname), os.path.join(dst_lora, fname))
        print("  Restored trained model files.")

else:
    for attempt in range(1, 4):
        result = subprocess.run(
            ['git', 'clone', 'https://github.com/munazat/UrduStack.git'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            break
        print(f"Clone failed (attempt {attempt}/3)")
        if attempt < 3:
            time.sleep(5)
    else:
        raise RuntimeError("Could not clone repo")
    os.chdir('UrduStack')

print(f"Working directory: {os.getcwd()}")

for f in ['tests/run_evaluation.py', 'tests/eval_dataset.csv',
           'playground.py', 'app/models/model_manager.py']:
    if not os.path.exists(f):
        raise RuntimeError(f"Missing: {f}")

import csv
with open('tests/eval_dataset.csv', 'r', encoding='utf-8') as f:
    row_count = len(list(csv.DictReader(f)))
print(f"Eval dataset: {row_count} examples")

log = subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True)
print(f"Latest commit: {log.stdout.strip()}")
print("Repo ready.")

In [ ]:
import os, shutil
from google.colab import files

model_dir = 'models/risk_lora'
os.makedirs(model_dir, exist_ok=True)
os.makedirs('models', exist_ok=True)

adapter_config = os.path.join(model_dir, 'adapter_config.json')
has_adapter = os.path.exists(adapter_config) and (
    os.path.exists(os.path.join(model_dir, 'adapter_model.safetensors')) or
    os.path.exists(os.path.join(model_dir, 'adapter_model.bin'))
)
has_temp = os.path.exists('models/temperature.txt') and open('models/temperature.txt').read().strip()

if has_adapter and has_temp:
    print("Model files already present. Skipping upload.")
    print(f"  temperature.txt: {open('models/temperature.txt').read().strip()}")
    print(f"  adapter files: {len(os.listdir(model_dir))} files")
else:
    print("=" * 60)
    print("UPLOAD TRAINED MODEL FILES")
    print("=" * 60)
    print()
    print("Select ALL files from your urdustack_model.zip:")
    print("  From risk_lora/: adapter_config.json, adapter_model.safetensors,")
    print("    tokenizer.json, tokenizer_config.json, sentencepiece.bpe.model")
    print("  From models/: temperature.txt")
    print()

    uploaded = files.upload()
    print(f"\nUploaded {len(uploaded)} file(s). Sorting...")

    for filename in uploaded:
        basename = os.path.basename(filename)
        if basename == 'temperature.txt':
            dst = 'models/temperature.txt'
        else:
            dst = os.path.join(model_dir, basename)
        if filename != dst:
            shutil.move(filename, dst)
            print(f"  {basename} -> {dst}")

    # Verify
    errors = []
    if not os.path.exists(os.path.join(model_dir, 'adapter_config.json')):
        errors.append("adapter_config.json missing")
    if not os.path.exists('models/temperature.txt'):
        errors.append("temperature.txt missing")

    if errors:
        print("\nERRORS: " + ", ".join(errors))
        print("Re-run this cell and upload the missing files.")
    else:
        print(f"\nAll files ready. Temperature: {open('models/temperature.txt').read().strip()}")

---
## Run Evaluations

Run the adversarial test first (quick), then the full evaluation.

In [ ]:
import subprocess, sys

os.chdir('/content/UrduStack')
sys.path.insert(0, '/content/UrduStack')

print("=" * 60)
print("ADVERSARIAL TEST — 12 red-team cases")
print("=" * 60)
print()

result = subprocess.run(
    [sys.executable, 'tests/run_evaluation.py', '--mode', 'adversarial'],
    capture_output=True, text=True, cwd='/content/UrduStack'
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)

In [ ]:
import subprocess, sys

os.chdir('/content/UrduStack')

print("=" * 60)
print("FULL EVALUATION — 107 examples")
print("=" * 60)
print()
print("This takes 5-10 minutes depending on GPU.")
print()

result = subprocess.run(
    [sys.executable, 'tests/run_evaluation.py', '--mode', 'full'],
    capture_output=True, text=True, cwd='/content/UrduStack'
)

print(result.stdout)
if result.stderr:
    # Filter out model loading noise
    lines = [l for l in result.stderr.split('\n')
             if 'Some weights' not in l and 'was not used' not in l
             and 'setting the pad_token' not in l]
    stderr_clean = '\n'.join(lines)
    if stderr_clean.strip():
        print("STDERR:", stderr_clean[-500:] if len(stderr_clean) > 500 else stderr_clean)

In [ ]:
import os, json
import pandas as pd
from google.colab import files

print("=" * 60)
print("RESULTS")
print("=" * 60)

# --- Adversarial metrics ---
adv_metrics_path = 'tests/eval_metrics_adversarial.json'
if os.path.exists(adv_metrics_path):
    with open(adv_metrics_path) as f:
        adv = json.load(f)
    print(f"\nAdversarial ({adv['total']} cases):")
    print(f"  Accuracy:  {adv['accuracy']:.4f}")
    print(f"  Precision: {adv['precision']:.4f}")
    print(f"  Recall:    {adv['recall']:.4f}")
    print(f"  F1:        {adv['f1']:.4f}")

# --- Full metrics ---
full_metrics_path = 'tests/eval_metrics_full.json'
if os.path.exists(full_metrics_path):
    with open(full_metrics_path) as f:
        full = json.load(f)
    print(f"\nFull Evaluation ({full['total']} examples):")
    print(f"  Accuracy:  {full['accuracy']:.4f}")
    print(f"  Precision: {full['precision']:.4f}")
    print(f"  Recall:    {full['recall']:.4f}")
    print(f"  F1:        {full['f1']:.4f}")
    print(f"\n  Per-category:")
    for cat, stats in full['per_category'].items():
        pct = stats['passed'] / stats['total'] * 100
        print(f"    {cat:12}: {stats['passed']}/{stats['total']} ({pct:.0f}%)")

# --- Show failures table ---
full_results_path = 'tests/eval_results_full.csv'
if os.path.exists(full_results_path):
    df = pd.read_csv(full_results_path)
    failures = df[df['correct'] == False]
    if len(failures) > 0:
        print(f"\nFailures ({len(failures)}):")
        display_cols = ['category', 'description', 'text', 'expected_label',
                        'predicted_label', 'risk_score', 'flagged_phrases']
        display(failures[display_cols].reset_index(drop=True))
    else:
        print("\nAll examples passed!")

# --- Download results ---
print("\nDownloading result files...")
for f in [adv_metrics_path, 'tests/eval_results_adversarial.csv',
          full_metrics_path, full_results_path]:
    if os.path.exists(f):
        try:
            files.download(f)
            print(f"  Downloaded: {f}")
        except Exception as e:
            print(f"  Failed: {f} — {e}")